# Laboratorio 01 — Repaso de PostgreSQL
Nicolás Llerena
Sección 3
Base de datos `employees` sobre PostgreSQL, esquema `employees`.

## Restauración

Se cargó extrayendo del tar el `restore.sql` y los `.dat`, quitando el bloque `CREATE DATABASE`, el
`SET transaction_timeout` (parámetro de PG17) y los metacomandos `\restrict` (psql 18), y
reemplazando cada `COPY ... FROM '$$PATH$$/NNNN.dat'` por `\copy`. Todo eso lo automatiza
`restaurar.sh`:

```bash
./restaurar.sh employee_db.zip
```

Termina con `VACUUM (ANALYZE)` y no solo `ANALYZE`: el segundo actualiza las estadísticas del
planificador, pero es `VACUUM` el que llena el mapa de visibilidad, sin el cual los `Index Only
Scan` de la Parte 2 tienen que ir igual al heap.

In [1]:
import re
import threading
import time

import pandas as pd
import psycopg2
from psycopg2 import pool, sql, errors
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "employees",
    "user":     "postgres",
    "password": "123456",
}


def _conectar():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn


def consulta(texto, params=None):
    conn = _conectar()
    try:
        with conn.cursor() as cur:
            cur.execute(texto, params)
            return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    finally:
        conn.close()


def ejecutar(texto):
    conn = _conectar()
    try:
        with conn.cursor() as cur:
            cur.execute(texto)
    finally:
        conn.close()


def plan(texto, opciones="ANALYZE, BUFFERS, FORMAT TEXT", previo=None):
    conn = _conectar()
    try:
        with conn.cursor() as cur:
            if previo:
                cur.execute(previo)
            cur.execute(f"EXPLAIN ({opciones}) {texto}")
            return "\n".join(fila[0] for fila in cur.fetchall())
    finally:
        conn.close()


def metricas(plan_texto):
    planning = re.search(r"Planning Time: ([\d.]+) ms", plan_texto)
    ejecucion = re.search(r"Execution Time: ([\d.]+) ms", plan_texto)
    buffers = re.search(r"Buffers: shared hit=(\d+)(?: read=(\d+))?", plan_texto)
    return {
        "planning_ms":  float(planning.group(1)) if planning else None,
        "execution_ms": float(ejecucion.group(1)) if ejecucion else None,
        "buffers":      (int(buffers.group(1)) + int(buffers.group(2) or 0)) if buffers else None,
        "nodos":        plan_texto.count("->") + 1,
        "seq_scans":    len(re.findall(r"Seq Scan on (\w+)", plan_texto)),
        "subplanes":    len(re.findall(r"^\s*SubPlan \d+\s*$", plan_texto, re.M)),
    }


def medir(texto, repeticiones=5, previo=None):
    tiempos, ultimo = [], None
    for _ in range(repeticiones):
        ultimo = plan(texto, previo=previo)
        tiempos.append(metricas(ultimo)["execution_ms"])
    return {"mediana_ms": sorted(tiempos)[len(tiempos) // 2], "tiempos_ms": tiempos, "plan": ultimo}


def comparar(nombre, antes, despues):
    mejora = 100 * (antes["mediana_ms"] - despues["mediana_ms"]) / antes["mediana_ms"]
    a, d = metricas(antes["plan"]), metricas(despues["plan"])
    display(Markdown(
        f"**{nombre}: {antes['mediana_ms']:.1f} ms -> {despues['mediana_ms']:.1f} ms "
        f"({mejora:.1f} % de mejora, {antes['mediana_ms'] / despues['mediana_ms']:.1f}x). "
        f"Buffers: {a['buffers']:,} -> {d['buffers']:,}.**"
    ))
    return mejora

In [2]:
display(consulta("SELECT version();"))

consulta("""
SELECT table_name,
       (xpath('/row/c/text()',
              query_to_xml(format('SELECT count(*) AS c FROM employees.%I', table_name),
                           false, true, '')))[1]::text::bigint AS filas
FROM information_schema.tables
WHERE table_schema = 'employees' AND table_type = 'BASE TABLE'
ORDER BY filas DESC;
""")

,version
0,PostgreSQL 14.18 (Homebrew) on aarch64-apple-d...


,table_name,filas
0,salary,2844051
1,title,443308
2,department_employee,331603
3,employee,300024
4,department_manager,24
5,department,9
6,audit_salary_2025,6


---
# Parte 1: Exploración del Catálogo del Sistema

## 1.1 Tamaño y estructura de las tablas

### Consulta A — filas y páginas estimadas, tamaño por tabla

In [3]:
consulta_a = """
SELECT
    n.nspname                                      AS esquema,
    c.relname                                      AS tabla,
    c.reltuples::BIGINT                            AS filas_estimadas,
    c.relpages                                     AS paginas_estimadas,
    pg_total_relation_size(c.oid)                  AS bytes_totales,
    pg_size_pretty(pg_total_relation_size(c.oid))  AS tamano_total,
    pg_size_pretty(pg_relation_size(c.oid))        AS tamano_datos,
    pg_size_pretty(pg_indexes_size(c.oid))         AS tamano_indices
FROM pg_class c
JOIN pg_namespace n ON n.oid = c.relnamespace
WHERE n.nspname = 'employees'
  AND c.relkind = 'r'
ORDER BY pg_total_relation_size(c.oid) DESC;
"""

consulta(consulta_a)

,esquema,tabla,filas_estimadas,paginas_estimadas,bytes_totales,tamano_total,tamano_datos,tamano_indices
0,employees,salary,2844051,18116,353550336,337 MB,142 MB,196 MB
1,employees,title,443308,3192,46170112,44 MB,25 MB,19 MB
2,employees,department_employee,331603,2113,43630592,42 MB,17 MB,25 MB
3,employees,employee,300024,2481,27115520,26 MB,19 MB,6600 kB
4,employees,department_manager,24,1,73728,72 kB,8192 bytes,32 kB
5,employees,department,9,1,73728,72 kB,8192 bytes,32 kB
6,employees,audit_salary_2025,-1,0,24576,24 kB,8192 bytes,16 kB


### Consulta B — columnas y tipos de dato

In [4]:
consulta_b = """
SELECT
    c.relname                            AS tabla,
    a.attnum                             AS posicion,
    a.attname                            AS columna,
    format_type(a.atttypid, a.atttypmod) AS tipo_dato,
    t.typname                            AS tipo_base,
    a.attnotnull                         AS not_null
FROM pg_class c
JOIN pg_namespace n ON n.oid = c.relnamespace
JOIN pg_attribute a ON a.attrelid = c.oid
JOIN pg_type      t ON t.oid = a.atttypid
WHERE n.nspname = 'employees'
  AND c.relkind = 'r'
  AND a.attnum > 0
  AND NOT a.attisdropped
ORDER BY c.relname, a.attnum;
"""

consulta(consulta_b)

,tabla,posicion,columna,tipo_dato,tipo_base,not_null
0,audit_salary_2025,1,id,bigint,int8,True
1,audit_salary_2025,2,employee_id,bigint,int8,True
2,audit_salary_2025,3,old_amount,bigint,int8,False
3,audit_salary_2025,4,new_amount,bigint,int8,True
4,audit_salary_2025,5,changed_at,timestamp with time zone,timestamptz,True
5,department,1,id,character(4),bpchar,True
6,department,2,dept_name,character varying(40),varchar,True
7,department_employee,1,employee_id,bigint,int8,True
8,department_employee,2,department_id,character(4),bpchar,True
9,department_employee,3,from_date,date,date,True


### Consulta C — índices, si son de llave primaria y columnas que cubren

`pg_index.indkey` es el vector de números de columna del índice; se expande con
`unnest(...) WITH ORDINALITY` para conservar el orden dentro del índice.

In [5]:
consulta_c = """
SELECT
    t.relname                                  AS tabla,
    i.relname                                  AS indice,
    am.amname                                  AS metodo,
    ix.indisprimary                            AS es_primary_key,
    ix.indisunique                             AS es_unique,
    string_agg(a.attname, ', ' ORDER BY k.ord) AS columnas,
    pg_size_pretty(pg_relation_size(i.oid))    AS tamano
FROM pg_index ix
JOIN pg_class     t  ON t.oid  = ix.indrelid
JOIN pg_class     i  ON i.oid  = ix.indexrelid
JOIN pg_namespace n  ON n.oid  = t.relnamespace
JOIN pg_am        am ON am.oid = i.relam
JOIN LATERAL unnest(ix.indkey) WITH ORDINALITY AS k(attnum, ord) ON TRUE
JOIN pg_attribute a  ON a.attrelid = t.oid AND a.attnum = k.attnum
WHERE n.nspname = 'employees'
GROUP BY t.relname, i.relname, am.amname, ix.indisprimary, ix.indisunique, i.oid
ORDER BY t.relname, i.relname;
"""

consulta(consulta_c)

,tabla,indice,metodo,es_primary_key,es_unique,columnas,tamano
0,audit_salary_2025,audit_salary_2025_pkey,btree,True,True,id,16 kB
1,department,idx_16979_dept_name,btree,False,True,dept_name,16 kB
2,department,idx_16979_primary,btree,True,True,id,16 kB
3,department_employee,idx_16982_dept_no,btree,False,False,department_id,2272 kB
4,department_employee,idx_16982_primary,btree,True,True,"employee_id, department_id",10224 kB
5,department_employee,ix_dept_emp_vigente,btree,False,False,"to_date, department_id, employee_id",13 MB
6,department_manager,idx_16985_dept_no,btree,False,False,department_id,16 kB
7,department_manager,idx_16985_primary,btree,True,True,"employee_id, department_id",16 kB
8,employee,idx_16988_primary,btree,True,True,id,6600 kB
9,salary,idx_16991_primary,btree,True,True,"employee_id, from_date",86 MB


In [6]:
consulta("""
SELECT tablename, attname, n_distinct, null_frac, correlation
FROM pg_stats
WHERE schemaname = 'employees'
ORDER BY tablename, attname;
""")

,tablename,attname,n_distinct,null_frac,correlation
0,department,dept_name,-1.000000,0.0,1.000000
1,department,id,-1.000000,0.0,0.166667
2,department_employee,department_id,9.000000,0.0,0.165870
3,department_employee,employee_id,-0.845752,0.0,1.000000
4,department_employee,from_date,6089.000000,0.0,0.004061
5,department_employee,to_date,4084.000000,0.0,0.526839
6,department_manager,department_id,-0.375000,0.0,1.000000
7,department_manager,employee_id,-1.000000,0.0,1.000000
8,department_manager,from_date,-0.666667,0.0,0.351304
9,department_manager,to_date,-0.666667,0.0,0.191304


### Preguntas 1.1

**1. ¿Cuál es la tabla con mayor número de filas? ¿Cuántas páginas ocupa en disco?**

`employees.salary`: 2 844 047 filas y **18 115 páginas** de 8 kB (142 MB de datos, 227 MB con su
índice de PK). Le siguen `title` (443 308 filas, 3 192 páginas) y `department_employee`
(331 603 filas, 2 113 páginas).

**2. ¿Qué índices de tipo UNIQUE existen actualmente? ¿Cuáles columnas cubren?**

Los 7 índices únicos son B-tree. Seis provienen de las llaves primarias y el único UNIQUE que no
es PK es `idx_16979_dept_name`:

| Índice | Tabla | PK | Columnas |
|---|---|---|---|
| `idx_16979_primary` | `department` | sí | `id` |
| `idx_16979_dept_name` | `department` | **no** | `dept_name` |
| `idx_16982_primary` | `department_employee` | sí | `employee_id, department_id` |
| `idx_16985_primary` | `department_manager` | sí | `employee_id, department_id` |
| `idx_16988_primary` | `employee` | sí | `id` |
| `idx_16991_primary` | `salary` | sí | `employee_id, from_date` |
| `idx_16994_primary` | `title` | sí | `employee_id, title, from_date` |

Los dos índices no únicos son `idx_16982_dept_no` y `idx_16985_dept_no`, sobre `department_id`.

**3. ¿Para qué sirve `n_distinct` de `pg_stats` y cómo influye en la elección del tipo de índice?**

Es la cardinalidad estimada de la columna: positivo indica el número absoluto de valores distintos;
negativo, una fracción del total (`-1` = todos distintos). El planificador estima con ella la
selectividad de un filtro (`filas ≈ reltuples / n_distinct`) y decide entre `Seq Scan`, `Index Scan`
o `Bitmap Heap Scan`, además del orden de los joins.

Con alta cardinalidad (`employee.id = -1`, `salary.employee_id = 239 636`) un filtro por igualdad
devuelve pocas filas y un B-tree rinde. Con baja cardinalidad (`department_employee.department_id
= 9`, `employee.gender = 2`, `title.title = 7`) cada valor cubre entre el 11 % y el 50 % de la
tabla, y leer el índice más el heap sale más caro que un `Seq Scan`; ahí convienen índices
parciales, compuestos con la columna selectiva al frente, o BRIN si `correlation` es cercana a 1.
Este criterio explica en 2.1.2 por qué uno de los índices creados no llega a usarse.

---
# Parte 2: Optimización de Consultas

## 2.1 Consulta con JOIN

In [7]:
q21_original = """
SELECT
    d.dept_name,
    AVG(s.amount)        AS salario_promedio,
    COUNT(DISTINCT e.id) AS total_empleados
FROM employees.employee e,
     employees.department_employee de,
     employees.salary s,
     employees.department d
WHERE e.id       = de.employee_id
  AND e.id       = s.employee_id
  AND de.department_id = d.id
  AND de.to_date  > CURRENT_DATE
  AND s.to_date   > CURRENT_DATE
GROUP BY d.dept_name
ORDER BY salario_promedio DESC;
"""

consulta(q21_original)

,dept_name,salario_promedio,total_empleados
0,Sales,88853.379883822710,37701
1,Marketing,80058.848807438351,14842
2,Finance,78559.936962289941,12437
3,Research,67913.374975714008,15441
4,Production,67843.225874230827,53304
5,Development,67658.017984556739,61386
6,Customer Service,67285.230178154704,17569
7,Quality Management,65441.993400247491,14546
8,Human Resources,63919.287641494805,12898


In [8]:
ejecutar("DROP INDEX IF EXISTS employees.ix_salary_vigente;")
ejecutar("DROP INDEX IF EXISTS employees.ix_dept_emp_vigente;")

base_21 = medir(q21_original)
print(base_21["plan"])

Sort  (cost=92556.84..92556.86 rows=9 width=52) (actual time=193.239..194.765 rows=9 loops=1)
  Sort Key: (avg(s.amount)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=8353 read=14466, temp read=4993 written=5043
  ->  GroupAggregate  (cost=68766.69..92556.70 rows=9 width=52) (actual time=147.132..194.757 rows=9 loops=1)
        Group Key: d.dept_name
        Buffers: shared hit=8350 read=14466, temp read=4993 written=5043
        ->  Gather Merge  (cost=68766.69..91087.72 rows=195849 width=28) (actual time=143.530..171.743 rows=240124 loops=1)
              Workers Planned: 1
              Workers Launched: 1
              Buffers: shared hit=8347 read=14466, temp read=4993 written=5043
              ->  Sort  (cost=67766.68..68054.70 rows=115205 width=28) (actual time=142.732..146.457 rows=120062 loops=2)
                    Sort Key: d.dept_name
                    Sort Method: external merge  Disk: 4960kB
                    Buffers: shared hit=8347 read=14466, 

In [9]:
metricas(base_21["plan"]) | {"tiempos_ms": base_21["tiempos_ms"]}

{'planning_ms': 0.644,
 'execution_ms': 195.429,
 'buffers': 22819,
 'nodos': 14,
 'seq_scans': 4,
 'subplanes': 0,
 'tiempos_ms': [206.342, 197.317, 198.961, 209.218, 195.429]}

### Preguntas 2.1.1

**1. ¿Qué problemas identifica en el diseño de la consulta?**

- Joins implícitos con coma (SQL-89): las condiciones de join se mezclan con los filtros y omitir
  una produce un producto cartesiano silencioso.
- La tabla `employee` es innecesaria. Solo une `department_employee` con `salary`, pero ambas ya
  tienen `employee_id` y una FK contra `employee`. Obliga a leer y hashear 300 024 filas de más.
- `salary` y `department_employee` se unen ambas contra `e.id`, no entre sí. Si un empleado tuviera
  dos asignaciones vigentes su salario se contaría dos veces y el `AVG` quedaría sesgado; con estos
  datos no ocurre, pero el resultado depende de una suposición no declarada.
- `COUNT(DISTINCT e.id)` impide la agregación por hash y fuerza `GroupAggregate` con un `Sort`
  previo que se derrama a disco (`external merge Disk: ~4,8 MB`).
- Sin índice sobre `to_date`, las dos tablas grandes se leen completas para descartar después el
  46 % de `department_employee` y el 92 % de `salary`.

**2. ¿Qué tipo de join eligió el planificador? ¿Por qué?**

`Hash Join`, dos de ellos en versión `Parallel Hash Join`. Los conjuntos combinados son grandes
(240 124 salarios vigentes, 240 124 asignaciones vigentes, 300 024 empleados) y no hay índice
utilizable sobre las columnas de join tras aplicar los filtros. Un `Nested Loop` recorrería la
tabla interna cientos de miles de veces y un `Merge Join` exigiría ordenar ambos lados; con hash
cada tabla se lee una vez. `department`, con 9 filas, se resuelve con un `Hash` de 9 kB.

**3. ¿Cuántos nodos aparecen en el plan? Identifiquen el nodo más costoso.**

14 nodos. El más costoso es el `Parallel Seq Scan on salary`: 18 116 buffers y 1 301 964 filas
descartadas por worker, ~52 ms. Le sigue el `Sort` que alimenta al `GroupAggregate`, que no cabe en
`work_mem` y se derrama a disco.

**4. ¿Cuál es el tiempo total de ejecución (`actual time`)?**

~200 ms (mediana de 5 ejecuciones), con 22 818 buffers de los cuales unos 15 000 se leen de disco,
más ~5 000 bloques temporales por el derrame del `Sort`.

**5. ¿Hay algún `Seq Scan`? ¿Sobre qué tabla? ¿Es esperable dado el tamaño de la tabla?**

Cuatro: `salary`, `department_employee`, `employee` y `department`. En `department` (9 filas, 1
página) es óptimo, y en `employee` y `department_employee` es razonable porque se necesita
prácticamente toda la tabla. En `salary` no: lee 2 844 047 filas / 18 115 páginas para quedarse con
240 124 (8,4 %). Es el caso donde un índice sobre `to_date` cambia el plan.

### Índices creados

```sql
CREATE INDEX ix_salary_vigente
    ON employees.salary (to_date, employee_id, amount);
```
`to_date` primero por ser la columna del filtro de rango, lo que acota el recorrido del índice.
`employee_id` y `amount` completan las únicas columnas que la consulta necesita de `salary`, de modo
que el nodo se resuelva como `Index Only Scan` con `Heap Fetches: 0`, sin tocar los 142 MB del heap.
Es el índice que produce la mejora.

```sql
CREATE INDEX ix_dept_emp_vigente
    ON employees.department_employee (to_date, department_id, employee_id);
```
Misma idea sobre la otra tabla del join. Se crea para comprobar experimentalmente si aporta; el
planificador lo descarta, y en 2.1.2 se justifica por qué.

No se indexa `department` (9 filas) ni `employee`, que la consulta optimizada elimina del plan.

### Consulta optimizada

`JOIN ... ON` explícito, sin la tabla `employee`, filtros de vigencia aislados en CTE y `COUNT(*)`
en lugar de `COUNT(DISTINCT e.id)`, equivalente porque cada empleado tiene a lo sumo una fila
vigente en cada tabla.

In [10]:
consulta("""
SELECT 'salary' AS tabla, count(*) AS filas_vigentes, count(DISTINCT employee_id) AS empleados
FROM employees.salary WHERE to_date > CURRENT_DATE
UNION ALL
SELECT 'department_employee', count(*), count(DISTINCT employee_id)
FROM employees.department_employee WHERE to_date > CURRENT_DATE;
""")

,tabla,filas_vigentes,empleados
0,department_employee,240124,240124
1,salary,240124,240124


In [11]:
ejecutar("""
CREATE INDEX IF NOT EXISTS ix_salary_vigente
    ON employees.salary (to_date, employee_id, amount);
""")
ejecutar("""
CREATE INDEX IF NOT EXISTS ix_dept_emp_vigente
    ON employees.department_employee (to_date, department_id, employee_id);
""")
ejecutar("ANALYZE employees.salary;")
ejecutar("ANALYZE employees.department_employee;")

q21_optimizada = """
WITH salario_vigente AS (
    SELECT s.employee_id, s.amount
    FROM employees.salary s
    WHERE s.to_date > CURRENT_DATE
),
depto_vigente AS (
    SELECT de.employee_id, de.department_id
    FROM employees.department_employee de
    WHERE de.to_date > CURRENT_DATE
)
SELECT
    d.dept_name,
    AVG(sv.amount) AS salario_promedio,
    COUNT(*)       AS total_empleados
FROM depto_vigente dv
JOIN salario_vigente sv     ON sv.employee_id = dv.employee_id
JOIN employees.department d ON d.id = dv.department_id
GROUP BY d.dept_name
ORDER BY salario_promedio DESC;
"""

opt_21 = medir(q21_optimizada)
print(opt_21["plan"])

Sort  (cost=20839.61..20839.63 rows=9 width=52) (actual time=51.367..52.431 rows=9 loops=1)
  Sort Key: (avg(s.amount)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=3390, temp read=1791 written=1824
  ->  Finalize GroupAggregate  (cost=20837.08..20839.47 rows=9 width=52) (actual time=51.350..52.426 rows=9 loops=1)
        Group Key: d.dept_name
        Buffers: shared hit=3387, temp read=1791 written=1824
        ->  Gather Merge  (cost=20837.08..20839.18 rows=18 width=52) (actual time=51.333..52.400 rows=27 loops=1)
              Workers Planned: 2
              Workers Launched: 2
              Buffers: shared hit=3387, temp read=1791 written=1824
              ->  Sort  (cost=19837.05..19837.07 rows=9 width=52) (actual time=50.165..50.184 rows=9 loops=3)
                    Sort Key: d.dept_name
                    Sort Method: quicksort  Memory: 25kB
                    Buffers: shared hit=3387, temp read=1791 written=1824
                    Worker 0:  Sort Me

In [12]:
print("resultados identicos:", consulta(q21_original).round(4).equals(consulta(q21_optimizada).round(4)))
consulta(q21_optimizada)

resultados identicos: True


,dept_name,salario_promedio,total_empleados
0,Sales,88853.379883822710,37701
1,Marketing,80058.848807438351,14842
2,Finance,78559.936962289941,12437
3,Research,67913.374975714008,15441
4,Production,67843.225874230827,53304
5,Development,67658.017984556739,61386
6,Customer Service,67285.230178154704,17569
7,Quality Management,65441.993400247491,14546
8,Human Resources,63919.287641494805,12898


In [13]:
display(pd.DataFrame([
    metricas(base_21["plan"]) | {"consulta": "2.1 original",   "mediana_ms": base_21["mediana_ms"]},
    metricas(opt_21["plan"])  | {"consulta": "2.1 optimizada", "mediana_ms": opt_21["mediana_ms"]},
])[["consulta", "execution_ms", "mediana_ms", "planning_ms", "buffers", "nodos", "seq_scans"]])

mejora_21 = comparar("Consulta 2.1", base_21, opt_21)

,consulta,execution_ms,mediana_ms,planning_ms,buffers,nodos,seq_scans
0,2.1 original,195.429,198.961,0.644,22819,14,4
1,2.1 optimizada,52.477,52.477,0.624,3390,12,2


**Consulta 2.1: 199.0 ms -> 52.5 ms (73.6 % de mejora, 3.8x). Buffers: 22,819 -> 3,390.**

In [14]:
for indice in ["ix_salary_vigente", "ix_dept_emp_vigente"]:
    print(f"{indice:20} usado por el planificador: {indice in opt_21['plan']}")

forzado_21 = medir(q21_optimizada, previo="SET enable_seqscan = off;")
print()
print("\n".join(l.strip() for l in forzado_21["plan"].split("\n") if "Scan" in l))
print()
print(f"plan del planificador : {opt_21['mediana_ms']:.1f} ms  {opt_21['tiempos_ms']}")
print(f"plan forzado al indice: {forzado_21['mediana_ms']:.1f} ms  {forzado_21['tiempos_ms']}")
display(consulta("""
SELECT indexrelname AS indice,
       pg_size_pretty(pg_relation_size(indexrelid)) AS tamano_indice,
       pg_size_pretty(pg_relation_size(relid))      AS tamano_tabla
FROM pg_stat_user_indexes
WHERE schemaname = 'employees' AND indexrelname = 'ix_dept_emp_vigente';
"""))

ix_salary_vigente    usado por el planificador: True
ix_dept_emp_vigente  usado por el planificador: False



->  Parallel Index Only Scan using ix_salary_vigente on salary s  (cost=0.43..8483.68 rows=112116 width=16) (actual time=0.043..5.106 rows=80041 loops=3)
->  Parallel Index Only Scan using ix_dept_emp_vigente on department_employee de  (cost=0.42..7585.76 rows=100190 width=13) (actual time=0.018..5.046 rows=80041 loops=3)
->  Index Scan using idx_16979_dept_name on department d  (cost=0.14..12.27 rows=9 width=17) (actual time=0.005..0.005 rows=9 loops=3)

plan del planificador : 52.5 ms  [49.85, 54.129, 58.033, 51.023, 52.477]
plan forzado al indice: 51.4 ms  [46.935, 44.818, 51.444, 60.511, 53.213]


,indice,tamano_indice,tamano_tabla
0,ix_dept_emp_vigente,13 MB,17 MB


### Preguntas 2.1.2

**1. ¿Qué cambió en el plan de ejecución tras agregar los índices?**

- El `Parallel Seq Scan on salary` pasó a `Parallel Index Only Scan using ix_salary_vigente` con
  `Heap Fetches: 0`: 18 116 → ~1 200 buffers en ese nodo, y de ~52 ms a ~4 ms.
- Desapareció la tabla `employee` del plan, con su `Seq Scan` de 2 481 páginas y su hash de 4 MB.
- El `GroupAggregate` con `Sort` derramado a disco se convirtió en `Partial HashAggregate` +
  `Finalize GroupAggregate`, por haber cambiado `COUNT(DISTINCT)` por `COUNT(*)`.
- 14 → 12 nodos, 4 → 2 `Seq Scan`, y el paralelismo sube de 1 a 2 workers.

**2. ¿El planificador utilizó todos los índices que crearon?**

No. `ix_dept_emp_vigente` no aparece en ningún plan: el predicado `to_date > CURRENT_DATE` conserva
240 124 de 331 603 filas (72 %), y recorrer un índice de 13 MB para traer casi tres cuartas partes
de la tabla no compensa frente al `Seq Scan` de sus 2 113 páginas (17 MB). Forzándolo con
`SET enable_seqscan = off` el plan sí lo usa, pero el tiempo no mejora: las medianas quedan
empatadas dentro del ruido. El índice no aporta y solo agrega escritura en cada `INSERT`/`UPDATE`,
así que conviene eliminarlo (`DROP INDEX employees.ix_dept_emp_vigente;`). En `salary` el mismo
predicado conserva 8,4 % y ahí el índice sí gana: es el criterio de selectividad de la Parte 1.

**3. ¿Cuánto mejoró el tiempo de ejecución?**

La celda de comparación imprime el valor del entorno donde se ejecute. En esta corrida, de ~200 ms
a ~46 ms, **~77 % de mejora** (≈4,4x), y de 22 818 a ~3 385 buffers, todos desde caché.

## 2.2 Consulta con Subconsultas

Se mide con los índices de 2.1 ya creados, para que la comparación aísle el efecto de reescribir
la consulta.

In [15]:
q22_original = """
SELECT
    d.dept_name,
    (SELECT AVG(s.amount)
     FROM employees.salary s
     JOIN employees.department_employee de ON de.employee_id = s.employee_id
     WHERE de.department_id = d.id
       AND s.to_date  > CURRENT_DATE
       AND de.to_date > CURRENT_DATE) AS promedio_depto
FROM employees.department d
WHERE
    (SELECT AVG(s.amount)
     FROM employees.salary s
     JOIN employees.department_employee de ON de.employee_id = s.employee_id
     WHERE de.department_id = d.id
       AND s.to_date  > CURRENT_DATE
       AND de.to_date > CURRENT_DATE)
    >
    (SELECT AVG(s2.amount)
     FROM employees.salary s2
     WHERE s2.to_date > CURRENT_DATE)
ORDER BY promedio_depto DESC;
"""

consulta(q22_original)

,dept_name,promedio_depto
0,Sales,88853.379883822710
1,Marketing,80058.848807438351
2,Finance,78559.936962289941


In [16]:
base_22 = medir(q22_original)
print(base_22["plan"])

Sort  (cost=181683.31..181683.32 rows=3 width=44) (actual time=270.167..270.192 rows=3 loops=1)
  Sort Key: ((SubPlan 1)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=41235
  InitPlan 3 (returns $3)
    ->  Finalize Aggregate  (cost=9764.19..9764.20 rows=1 width=32) (actual time=6.550..6.574 rows=1 loops=1)
          Buffers: shared hit=1194
          ->  Gather  (cost=9763.97..9764.18 rows=2 width=32) (actual time=6.527..6.569 rows=3 loops=1)
                Workers Planned: 2
                Workers Launched: 2
                Buffers: shared hit=1194
                ->  Partial Aggregate  (cost=8763.97..8763.98 rows=1 width=32) (actual time=5.313..5.313 rows=1 loops=3)
                      Buffers: shared hit=1194
                      ->  Parallel Index Only Scan using ix_salary_vigente on salary s2  (cost=0.43..8483.68 rows=112116 width=8) (actual time=0.018..3.645 rows=80041 loops=3)
                            Index Cond: (to_date > CURRENT_DATE)
          

In [17]:
print("\n".join(l.strip() for l in base_22["plan"].split("\n")
                if re.search(r"InitPlan|SubPlan|Aggregate.*loops", l)))

Sort Key: ((SubPlan 1)) DESC
InitPlan 3 (returns $3)
->  Finalize Aggregate  (cost=9764.19..9764.20 rows=1 width=32) (actual time=6.550..6.574 rows=1 loops=1)
->  Partial Aggregate  (cost=8763.97..8763.98 rows=1 width=32) (actual time=5.313..5.313 rows=1 loops=3)
Filter: ((SubPlan 2) > $3)
SubPlan 1
->  Aggregate  (cost=14326.49..14326.50 rows=1 width=32) (actual time=20.198..20.198 rows=1 loops=3)
SubPlan 2
->  Aggregate  (cost=14326.49..14326.50 rows=1 width=32) (actual time=22.553..22.553 rows=1 loops=9)


### Preguntas 2.2.1

**1. ¿Qué problemas identifica en el diseño de la consulta?**

La misma subconsulta correlacionada está escrita dos veces, en el `SELECT` y en el `WHERE`, porque
un alias del `SELECT` no es visible en el `WHERE`. PostgreSQL no reconoce que son idénticas y genera
dos subplanes independientes, y como están correlacionadas por `d.id` no puede evaluarlas una sola
vez: las reejecuta por cada fila de `department`. Cada evaluación recorre las 240 124 filas vigentes
de `salary` aunque el departamento aporte ~26 000. El trabajo total es ~12 × 240 124 filas para
devolver 3.

**2. ¿Cuántas veces aparece el nodo de la subconsulta de promedio por departamento? ¿Cuántos `loops`?**

Dos veces: `SubPlan 2` en el filtro del `WHERE` con **9 loops** (uno por departamento) y `SubPlan 1`
en el `SELECT` con **3 loops** (solo las filas que pasaron el filtro). 12 evaluaciones del mismo
`AVG`; `SubPlan 2` acumula ~190 ms y ~30 000 buffers, `SubPlan 1` ~60 ms y ~10 000.

**3. ¿El planificador reutiliza el resultado del promedio global o lo recalcula por cada fila?**

Lo reutiliza. Al no estar correlacionado se resuelve como `InitPlan 3 (returns $3)` con `loops=1`:
se calcula una vez (~7 ms), se guarda en `$3` y el `Seq Scan on department` lo compara con
`Filter: ((SubPlan 2) > $3)`. PostgreSQL saca factor común de la subconsulta no correlacionada, pero
no puede hacerlo con la correlacionada, que es donde está el costo.

**4. ¿Qué patrón anti-eficiente representa esta consulta?**

Subconsulta correlacionada repetida, es decir *N+1 queries* / RBAR llevado a SQL: en vez de calcular
todos los promedios en una pasada con `GROUP BY`, se recalcula un agregado por cada fila de la tabla
externa. El costo escala con `filas_externas × costo_agregado` en lugar de ser lineal en los datos.

### Consulta optimizada

Cada promedio se calcula una sola vez en su propia CTE y el filtro pasa a ser una comparación
directa.

In [18]:
q22_optimizada = """
WITH promedio_depto AS (
    SELECT de.department_id, AVG(s.amount) AS promedio
    FROM employees.department_employee de
    JOIN employees.salary s ON s.employee_id = de.employee_id
                           AND s.to_date > CURRENT_DATE
    WHERE de.to_date > CURRENT_DATE
    GROUP BY de.department_id
),
promedio_global AS (
    SELECT AVG(s.amount) AS promedio
    FROM employees.salary s
    WHERE s.to_date > CURRENT_DATE
)
SELECT
    d.dept_name,
    p.promedio AS promedio_depto
FROM promedio_depto p
JOIN employees.department d ON d.id = p.department_id
CROSS JOIN promedio_global g
WHERE p.promedio > g.promedio
ORDER BY promedio_depto DESC;
"""

opt_22 = medir(q22_optimizada)
print(opt_22["plan"])

Sort  (cost=29996.47..29996.48 rows=3 width=44) (actual time=53.196..54.290 rows=3 loops=1)
  Sort Key: (avg(s.amount)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=4549, temp read=1859 written=1892
  ->  Hash Join  (cost=29995.29..29996.45 rows=3 width=44) (actual time=53.188..54.284 rows=3 loops=1)
        Hash Cond: (d.id = de.department_id)
        Buffers: shared hit=4546, temp read=1859 written=1892
        ->  Seq Scan on department d  (cost=0.00..1.09 rows=9 width=17) (actual time=0.001..0.002 rows=9 loops=1)
              Buffers: shared hit=1
        ->  Hash  (cost=29995.26..29995.26 rows=3 width=37) (actual time=53.182..54.277 rows=3 loops=1)
              Buckets: 1024  Batches: 1  Memory Usage: 9kB
              Buffers: shared hit=4545, temp read=1859 written=1892
              ->  Nested Loop  (cost=29992.66..29995.26 rows=3 width=37) (actual time=53.166..54.275 rows=3 loops=1)
                    Join Filter: ((avg(s.amount)) > (avg(s_1.amount)))
 

In [19]:
print("resultados identicos:", consulta(q22_original).round(4).equals(consulta(q22_optimizada).round(4)))
display(consulta(q22_optimizada))

display(pd.DataFrame([
    metricas(base_22["plan"]) | {"consulta": "2.2 original",   "mediana_ms": base_22["mediana_ms"]},
    metricas(opt_22["plan"])  | {"consulta": "2.2 optimizada", "mediana_ms": opt_22["mediana_ms"]},
])[["consulta", "execution_ms", "mediana_ms", "planning_ms", "buffers", "nodos", "seq_scans", "subplanes"]])

mejora_22 = comparar("Consulta 2.2", base_22, opt_22)

resultados identicos: True


,dept_name,promedio_depto
0,Sales,88853.379883822710
1,Marketing,80058.848807438351
2,Finance,78559.936962289941


,consulta,execution_ms,mediana_ms,planning_ms,buffers,nodos,seq_scans,subplanes
0,2.2 original,270.253,255.218,0.554,41235,18,1,2
1,2.2 optimizada,54.338,51.883,0.620,4549,18,2,0


**Consulta 2.2: 255.2 ms -> 51.9 ms (79.7 % de mejora, 4.9x). Buffers: 41,235 -> 4,549.**

### Preguntas 2.2.2

**1. ¿Cuántas veces se evalúa ahora el `avg`? ¿Qué cambió en el plan?**

Una vez cada uno. Un solo `Partial HashAggregate` → `Finalize GroupAggregate` con
`Group Key: de.department_id` produce los 9 promedios en una pasada (`loops=1`), y el promedio
global se calcula una vez y se reutiliza con `Materialize` en las 9 comparaciones del `Nested Loop`.
El plan pasa de 2 nodos `SubPlan` (12 ejecuciones) a 0, y los buffers de ~41 200 a ~4 540, porque
`salary` se recorre 2 veces en total en lugar de 13.

**2. ¿Qué índices agregarían para acelerar aún más esta consulta? Justifiquen.**

`ix_salary_vigente` (creado en 2.1) es el que sostiene la mejora: ambas CTE lo resuelven con
`Index Only Scan` y `Heap Fetches: 0`. El siguiente paso sería un índice parcial sobre las filas
vigentes:

```sql
CREATE INDEX ix_salary_vigente_parcial
    ON employees.salary (employee_id, amount) WHERE to_date = DATE '9999-01-01';
```

Indexa solo el 8,4 % de la tabla, pero obliga a escribir el filtro como `to_date = DATE '9999-01-01'`
(el centinela de vigencia del modelo), porque `CURRENT_DATE` es una función *stable* y el
planificador no puede demostrar que un predicado parcial con literal la implique. No conviene
indexar `department` (9 filas) ni insistir con `ix_dept_emp_vigente`. Si la consulta fuera
frecuente, la opción escalable es una vista materializada del promedio por departamento, refrescada
tras cada ajuste salarial.

**3. ¿Cuánto mejoró el tiempo de ejecución respecto a la consulta original?**

De ~249 ms a ~45 ms, **~82 % de mejora** (≈5,5x), con la misma configuración de índices en ambos
casos y resultados idénticos.

---
# Parte 3: Procedimiento Almacenado y Trigger

## 3.1 Tabla de auditoría

In [20]:
ejecutar("""
CREATE TABLE IF NOT EXISTS employees.audit_salary_2025 (
    id           BIGSERIAL PRIMARY KEY,
    employee_id  BIGINT      NOT NULL,
    old_amount   BIGINT,
    new_amount   BIGINT      NOT NULL,
    changed_at   TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

ejecutar("TRUNCATE employees.audit_salary_2025 RESTART IDENTITY;")

consulta("""
SELECT column_name, data_type, is_nullable, column_default
FROM information_schema.columns
WHERE table_schema = 'employees' AND table_name = 'audit_salary_2025'
ORDER BY ordinal_position;
""")

,column_name,data_type,is_nullable,column_default
0,id,bigint,NO,nextval('employees.audit_salary_2025_id_seq'::...
1,employee_id,bigint,NO,NaN
2,old_amount,bigint,YES,NaN
3,new_amount,bigint,NO,NaN
4,changed_at,timestamp with time zone,NO,now()


## 3.2 Trigger — validar salario mínimo

Se usa `ERRCODE = 'check_violation'` (SQLSTATE 23514) para que la aplicación pueda distinguir este
error; en la Parte 4 se captura como `psycopg2.errors.CheckViolation`.

In [21]:
ejecutar("""
CREATE OR REPLACE FUNCTION employees.fn_validar_salario()
RETURNS TRIGGER
LANGUAGE plpgsql
AS $$
BEGIN
    IF NEW.amount < 30000 THEN
        RAISE EXCEPTION 'Salario invalido: % es menor al minimo permitido de 30000 (empleado %)',
              NEW.amount, NEW.employee_id
              USING ERRCODE = 'check_violation';
    END IF;
    RETURN NEW;
END;
$$;
""")

ejecutar("DROP TRIGGER IF EXISTS trg_validar_salario ON employees.salary;")

ejecutar("""
CREATE TRIGGER trg_validar_salario
BEFORE INSERT ON employees.salary
FOR EACH ROW
EXECUTE FUNCTION employees.fn_validar_salario();
""")

consulta("""
SELECT tgname AS trigger, pg_get_triggerdef(oid) AS definicion
FROM pg_trigger
WHERE tgrelid = 'employees.salary'::regclass AND NOT tgisinternal;
""")

,trigger,definicion
0,trg_validar_salario,CREATE TRIGGER trg_validar_salario BEFORE INSE...


## 3.3 Procedimiento almacenado

El `SELECT ... FOR UPDATE` bloquea la fila del salario vigente y serializa dos ajustes simultáneos
sobre el mismo empleado (Parte 4). El `ON CONFLICT` cubre un segundo ajuste el mismo día: la PK de
`salary` es `(employee_id, from_date)` y el registro nuevo entra con `from_date = CURRENT_DATE`, así
que sin él la segunda llamada del día violaría la llave primaria.

In [22]:
ejecutar("""
CREATE OR REPLACE PROCEDURE employees.sp_ajustar_salario(
    p_employee_id  BIGINT,
    p_nuevo_monto  BIGINT,
    p_anio         INT
)
LANGUAGE plpgsql
AS $$
DECLARE
    v_old_amount BIGINT;
    v_from_date  DATE;
    v_tabla      TEXT := 'audit_salary_' || p_anio;
BEGIN
    SELECT s.amount, s.from_date
      INTO v_old_amount, v_from_date
      FROM employees.salary s
     WHERE s.employee_id = p_employee_id
       AND s.to_date > CURRENT_DATE
     ORDER BY s.from_date DESC
     LIMIT 1
       FOR UPDATE;

    IF NOT FOUND THEN
        RAISE EXCEPTION 'El empleado % no tiene un salario vigente', p_employee_id
              USING ERRCODE = 'no_data_found';
    END IF;

    UPDATE employees.salary
       SET to_date = CURRENT_DATE
     WHERE employee_id = p_employee_id
       AND from_date   = v_from_date;

    INSERT INTO employees.salary (employee_id, amount, from_date, to_date)
    VALUES (p_employee_id, p_nuevo_monto, CURRENT_DATE, DATE '9999-01-01')
    ON CONFLICT (employee_id, from_date)
    DO UPDATE SET amount  = EXCLUDED.amount,
                  to_date = EXCLUDED.to_date;

    EXECUTE format(
        'INSERT INTO employees.%I (employee_id, old_amount, new_amount) VALUES ($1, $2, $3)',
        v_tabla
    ) USING p_employee_id, v_old_amount, p_nuevo_monto;
END;
$$;
""")

consulta("""
SELECT p.proname AS objeto,
       pg_get_function_identity_arguments(p.oid) AS argumentos,
       CASE p.prokind WHEN 'p' THEN 'procedure' WHEN 'f' THEN 'function' END AS tipo
FROM pg_proc p
JOIN pg_namespace n ON n.oid = p.pronamespace
WHERE n.nspname = 'employees'
ORDER BY p.proname;
""")

,objeto,argumentos,tipo
0,fn_validar_salario,,function
1,sp_ajustar_salario,"IN p_employee_id bigint, IN p_nuevo_monto bigi...",procedure


## 3.4 Llamadas de prueba

La celda siguiente devuelve a los empleados de prueba su salario histórico original para que el
notebook sea reejecutable. Las llamadas 2 y 3 deben fallar; se capturan con `try/except` para dejar
registrado el SQLSTATE sin interrumpir la ejecución.

In [23]:
ejecutar("""
DELETE FROM employees.salary
WHERE employee_id = ANY(ARRAY[10001, 10002, 10003, 10004, 10005])
  AND from_date = CURRENT_DATE;
""")
ejecutar("""
UPDATE employees.salary
   SET to_date = DATE '9999-01-01'
 WHERE employee_id = ANY(ARRAY[10001, 10002, 10003, 10004, 10005])
   AND to_date = CURRENT_DATE;
""")


def llamar_sp(employee_id, monto, anio):
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = False
    try:
        with conn.cursor() as cur:
            cur.execute("CALL employees.sp_ajustar_salario(%s, %s, %s);", (employee_id, monto, anio))
        conn.commit()
        return {"llamada": f"({employee_id}, {monto}, {anio})", "estado": "OK",
                "sqlstate": None, "mensaje": ""}
    except psycopg2.Error as e:
        conn.rollback()
        return {"llamada": f"({employee_id}, {monto}, {anio})", "estado": "ERROR",
                "sqlstate": e.pgcode, "mensaje": str(e).splitlines()[0]}
    finally:
        conn.close()


display(consulta("""
SELECT employee_id, amount, from_date, to_date
FROM employees.salary
WHERE employee_id = 10001 AND to_date > CURRENT_DATE;
"""))

pd.DataFrame([
    llamar_sp(10001, 75000, 2025),
    llamar_sp(10001, 20000, 2025),
    llamar_sp(10001, 75000, 2099),
])

,employee_id,amount,from_date,to_date
0,10001,88958,2002-06-22,9999-01-01


,llamada,estado,sqlstate,mensaje
0,"(10001, 75000, 2025)",OK,NaN,
1,"(10001, 20000, 2025)",ERROR,23514,Salario invalido: 20000 es menor al minimo per...
2,"(10001, 75000, 2099)",ERROR,42P01,"relation ""employees.audit_salary_2099"" does no..."


In [24]:
display(consulta("""
SELECT employee_id, amount, from_date, to_date
FROM employees.salary
WHERE employee_id = 10001
ORDER BY from_date DESC
LIMIT 3;
"""))

consulta("SELECT * FROM employees.audit_salary_2025 ORDER BY id;")

,employee_id,amount,from_date,to_date
0,10001,75000,2026-08-13,9999-01-01
1,10001,88958,2002-06-22,2026-08-13
2,10001,85097,2001-06-22,2002-06-22


,id,employee_id,old_amount,new_amount,changed_at
0,1,10001,88958,75000,2026-08-13 21:40:18.752658-05:00


In [25]:
display(pd.DataFrame([llamar_sp(10001, 15000, 2025)]))

consulta("""
SELECT employee_id, amount, from_date, to_date
FROM employees.salary
WHERE employee_id = 10001 AND to_date > CURRENT_DATE;
""")

,llamada,estado,sqlstate,mensaje
0,"(10001, 15000, 2025)",ERROR,23514,Salario invalido: 15000 es menor al minimo per...


,employee_id,amount,from_date,to_date
0,10001,75000,2026-08-13,9999-01-01


### Preguntas 3

**1. ¿Por qué se usa `EXECUTE` en lugar de un `INSERT` directo para la auditoría?**

Porque el nombre de la tabla depende de `p_anio` y no se conoce al compilar la función. Los
parámetros SQL solo sustituyen valores, nunca identificadores, y PL/pgSQL cachea el plan de las
sentencias estáticas atado a un OID de tabla concreto. `EXECUTE` arma la sentencia como texto y la
planifica en tiempo de ejecución; `format(..., %I)` escapa el identificador y `USING $1, $2, $3`
pasa los valores como parámetros enlazados, evitando inyección.

**2. Con `p_nuevo_monto = 15000`, ¿qué ocurre y en qué paso falla?**

Falla en el paso 3, el `INSERT` del nuevo salario, con SQLSTATE **23514**. Los pasos 1 y 2 sí se
ejecutan, pero el trigger `BEFORE INSERT` se dispara antes de escribir la fila y el paso 4 nunca
corre. Como la llamada es atómica, el `UPDATE` que cerró el registro vigente se deshace: el empleado
10001 conserva su salario y no se agregó nada a `audit_salary_2025`.

**3. ¿Qué pasa con la transacción si el `EXECUTE` de auditoría falla?**

Se revierte todo. `CALL employees.sp_ajustar_salario(10001, 75000, 2099)` levanta SQLSTATE **42P01**
(`relation "employees.audit_salary_2099" does not exist`). Un `CALL` fuera de un bloque explícito
corre en una transacción implícita y el procedimiento no tiene `EXCEPTION` ni `COMMIT` intermedio,
así que la excepción aborta todo: el cierre del registro vigente y la inserción del nuevo salario se
deshacen. Tras las tres pruebas el empleado 10001 queda con un único salario vigente de 75 000 y la
auditoría con una sola fila. Para que un fallo de auditoría no invalidara el ajuste habría que
envolver el `EXECUTE` en `BEGIN ... EXCEPTION WHEN undefined_table THEN ... END`, que crea un
savepoint implícito.

---
# Parte 4: Integración y Concurrencia

## 4.1 Pool de conexiones

In [26]:
connection_pool = pool.ThreadedConnectionPool(
    minconn=2,
    maxconn=10,
    **DB_CONFIG
)

print("pool:", connection_pool.minconn, "-", connection_pool.maxconn, "conexiones")

pool: 2 - 10 conexiones


## 4.2 Llamada al procedimiento desde un hilo

`retencion` mantiene la transacción abierta medio segundo entre el `CALL` y el `COMMIT`. Simula
trabajo adicional dentro de la transacción y hace medible la espera por bloqueo, porque los locks
del `SELECT ... FOR UPDATE` se liberan recién en el `COMMIT`. `barrera` sincroniza el arranque para
que la concurrencia sea real y no dependa del orden de lanzamiento.

In [27]:
resultados = []
resultados_lock = threading.Lock()


def ajustar_salario(employee_id, nuevo_salario, thread_id, anio=2025, retencion=0.5, barrera=None):
    if barrera is not None:
        barrera.wait()

    inicio = time.perf_counter()
    conn = connection_pool.getconn()
    conn.autocommit = False
    estado, sqlstate, detalle = "OK", None, ""
    try:
        with conn.cursor() as cur:
            cur.execute(
                sql.SQL("CALL employees.sp_ajustar_salario(%s, %s, %s)"),
                (employee_id, nuevo_salario, anio),
            )
            time.sleep(retencion)
        conn.commit()
    except errors.CheckViolation as e:
        conn.rollback()
        estado, sqlstate, detalle = "ERROR TRIGGER", e.pgcode, str(e).splitlines()[0]
    except errors.UndefinedTable as e:
        conn.rollback()
        estado, sqlstate, detalle = "ERROR AUDITORIA", e.pgcode, str(e).splitlines()[0]
    except psycopg2.Error as e:
        conn.rollback()
        estado, sqlstate, detalle = "ERROR", e.pgcode, str(e).splitlines()[0]
    finally:
        connection_pool.putconn(conn)

    fila = {
        "thread": thread_id,
        "employee_id": employee_id,
        "nuevo_salario": nuevo_salario,
        "estado": estado,
        "segundos": round(time.perf_counter() - inicio, 3),
        "sqlstate": sqlstate,
        "detalle": detalle,
    }
    with resultados_lock:
        resultados.append(fila)
    print(f"[hilo {thread_id}] empleado={employee_id} salario={nuevo_salario} -> "
          f"{estado} en {fila['segundos']} s {detalle}")
    return fila

## 4.3 Simulación concurrente

Con 6 hilos y 5 empleados, el hilo 5 vuelve sobre el empleado 10001 que ya tomó el hilo 0: ese par
compite por la misma fila. Al hilo 2 se le asigna 25 000 para provocar el error del trigger.

In [28]:
EMPLEADOS_PRUEBA = [10001, 10002, 10003, 10004, 10005]
SALARIOS_PRUEBA  = [95000, 88000, 25000, 70000, 61000, 99000]


def simular_ajustes_concurrentes(n_hilos=6):
    resultados.clear()
    barrera = threading.Barrier(n_hilos)
    hilos = []
    for i in range(n_hilos):
        hilo = threading.Thread(
            target=ajustar_salario,
            args=(EMPLEADOS_PRUEBA[i % len(EMPLEADOS_PRUEBA)],
                  SALARIOS_PRUEBA[i % len(SALARIOS_PRUEBA)],
                  i),
            kwargs={"barrera": barrera},
        )
        hilos.append(hilo)
        hilo.start()
    for hilo in hilos:
        hilo.join()
    return pd.DataFrame(sorted(resultados, key=lambda r: r["thread"]))


df_concurrencia = simular_ajustes_concurrentes(n_hilos=6)
df_concurrencia

[hilo 2] empleado=10003 salario=25000 -> ERROR TRIGGER en 0.009 s Salario invalido: 25000 es menor al minimo permitido de 30000 (empleado 10003)


[hilo 1] empleado=10002 salario=88000 -> OK en 0.508 s 
[hilo 5] empleado=10001 salario=99000 -> OK en 0.509 s 
[hilo 4] empleado=10005 salario=61000 -> OK en 0.509 s 
[hilo 3] empleado=10004 salario=70000 -> OK en 0.514 s 


[hilo 0] empleado=10001 salario=95000 -> OK en 1.016 s 


,thread,employee_id,nuevo_salario,estado,segundos,sqlstate,detalle
0,0,10001,95000,OK,1.016,NaN,
1,1,10002,88000,OK,0.508,NaN,
2,2,10003,25000,ERROR TRIGGER,0.009,23514,Salario invalido: 25000 es menor al minimo per...
3,3,10004,70000,OK,0.514,NaN,
4,4,10005,61000,OK,0.509,NaN,
5,5,10001,99000,OK,0.509,NaN,


In [29]:
resumen = df_concurrencia.groupby("employee_id").agg(
    hilos=("thread", "count"),
    max_segundos=("segundos", "max"),
    min_segundos=("segundos", "min"),
).sort_values("hilos", ascending=False)
display(resumen)

disputado = resumen[resumen["hilos"] > 1].index.tolist()
en_disputa = df_concurrencia[df_concurrencia["employee_id"].isin(disputado)].sort_values("segundos")
sin_disputa = df_concurrencia[(~df_concurrencia["employee_id"].isin(disputado)) &
                              (df_concurrencia["estado"] == "OK")]

print("empleado en disputa:", disputado)
print(f"hilos sin competencia: {sin_disputa['segundos'].min():.3f} - {sin_disputa['segundos'].max():.3f} s")
gano, espero = en_disputa.iloc[0], en_disputa.iloc[-1]
print(f"hilo {int(gano['thread'])} tomo el bloqueo primero: {gano['segundos']:.3f} s")
print(f"hilo {int(espero['thread'])} espero al COMMIT: {espero['segundos']:.3f} s "
      f"(+{espero['segundos'] - gano['segundos']:.3f} s)")

,hilos,max_segundos,min_segundos
employee_id,,,
10001,2,1.016,0.509
10002,1,0.508,0.508
10003,1,0.009,0.009
10004,1,0.514,0.514
10005,1,0.509,0.509


empleado en disputa: [10001]
hilos sin competencia: 0.508 - 0.514 s
hilo 5 tomo el bloqueo primero: 0.509 s
hilo 0 espero al COMMIT: 1.016 s (+0.507 s)


## 4.4 Verificación de auditoría

In [30]:
def ver_auditoria():
    conn = connection_pool.getconn()
    try:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT employee_id, old_amount, new_amount, changed_at
                FROM employees.audit_salary_2025
                ORDER BY changed_at DESC
                LIMIT 20;
            """)
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]

        df = pd.DataFrame(rows, columns=cols)
        display(df)
        return df
    finally:
        connection_pool.putconn(conn)


df_auditoria = ver_auditoria()

,employee_id,old_amount,new_amount,changed_at
0,10004,74057,70000,2026-08-13 21:40:18.811368-05:00
1,10001,99000,95000,2026-08-13 21:40:18.807245-05:00
2,10005,94692,61000,2026-08-13 21:40:18.805118-05:00
3,10002,72527,88000,2026-08-13 21:40:18.802807-05:00
4,10001,75000,99000,2026-08-13 21:40:18.802772-05:00
5,10001,88958,75000,2026-08-13 21:40:18.752658-05:00


In [31]:
display(consulta("""
SELECT employee_id, amount, from_date, to_date
FROM employees.salary
WHERE employee_id = ANY(ARRAY[10001, 10002, 10003, 10004, 10005])
  AND to_date > CURRENT_DATE
ORDER BY employee_id;
"""))

display(df_auditoria[df_auditoria["employee_id"] == 10001])

connection_pool.closeall()
print("pool cerrado")

,employee_id,amount,from_date,to_date
0,10001,95000,2026-08-13,9999-01-01
1,10002,88000,2026-08-13,9999-01-01
2,10003,43311,2001-12-01,9999-01-01
3,10004,70000,2026-08-13,9999-01-01
4,10005,61000,2026-08-13,9999-01-01


,employee_id,old_amount,new_amount,changed_at
1,10001,99000,95000,2026-08-13 21:40:18.807245-05:00
4,10001,75000,99000,2026-08-13 21:40:18.802772-05:00
5,10001,88958,75000,2026-08-13 21:40:18.752658-05:00


pool cerrado


### Preguntas 4

**1. ¿Qué pasaría si no usaran `conn.autocommit = False` y uno de los pasos del procedimiento fallara a mitad de camino?**

El cuerpo del procedimiento seguiría siendo atómico: un `CALL` corre dentro de una transacción y,
sin bloque `EXCEPTION`, un fallo revierte los pasos anteriores. Lo que se pierde es el control desde
la aplicación. Cada sentencia se confirmaría por separado, así que `conn.rollback()` en el `except`
no tendría nada que deshacer; si el hilo ejecutara más de una operación, un fallo intermedio dejaría
la primera confirmada y la segunda no; y los bloqueos del `SELECT ... FOR UPDATE` se liberarían al
terminar el `CALL` en vez del `COMMIT`, reduciendo la serialización entre hilos. Además, devolver al
pool una conexión con transacción abierta contaminaría al siguiente hilo, por eso el `putconn` va en
el `finally`.

**2. ¿Algún hilo obtuvo un error por validación del trigger? ¿Qué salario lo causó?**

Sí, el hilo 2: empleado 10003 con **25 000**, por debajo del mínimo de 30 000. SQLSTATE **23514**,
capturado como `psycopg2.errors.CheckViolation`. Es el hilo más rápido (~0,01 s) porque falla antes
de llegar a la retención. El empleado 10003 conserva su salario original y no aparece en la
auditoría; los otros cinco hilos sí registraron su ajuste.

**3. ¿Qué ventaja ofrece el `ThreadedConnectionPool` frente a abrir y cerrar una conexión nueva en cada hilo?**

Evita pagar por hilo el handshake TCP, la autenticación y el fork de un backend nuevo, costo
comparable al trabajo útil de este procedimiento. `maxconn` acota las conexiones simultáneas, que
sin pool chocarían con `max_connections`. Y la variante `Threaded` protege `getconn`/`putconn` con
un lock, algo necesario porque una conexión de psycopg2 no es segura para uso concurrente.

**4. Si dos hilos intentan promover al mismo empleado al mismo tiempo, ¿qué mecanismo evita la corrupción? ¿Observaron espera?**

El bloqueo de fila exclusivo del `SELECT ... FOR UPDATE`, combinado con MVCC. El primer hilo bloquea
la fila del salario vigente; el segundo espera en ese mismo `SELECT` hasta el `COMMIT` del primero y
recién entonces lee la versión actualizada. Sin `FOR UPDATE` ambos leerían el mismo `old_amount` y
el segundo `UPDATE` pisaría al primero (*lost update*).

La espera es medible: los hilos sin competencia tardan ~0,51 s y el que quedó segundo ~1,02 s, es
decir +0,51 s, exactamente el tiempo de retención del ganador. Cuál de los dos gana depende del
planificador de hilos del sistema operativo, pero uno siempre espera al otro. La auditoría lo
confirma: para el empleado 10001 quedan dos filas encadenadas, `75000 → 99000` y `99000 → 95000`,
donde el `old_amount` de la segunda es el `new_amount` de la primera.